# CareScan — train the YOLO symbol detector

Data is kept on Colab's fast local disk (`/content`); only training results go to Drive.
After a disconnect, re-run sections 1–2 before training.

Run this in **Google Colab** with a GPU: *Runtime → Change runtime type → T4 GPU*.

Steps:
1. Connect Google Drive and install packages
2. Download datasets from Roboflow and merge them
3. Split (augmented copies kept together) and check class balance
4. Train YOLOv8 (transfer learning from COCO weights)
5. Evaluate on the test set (mAP@0.5, precision, recall, confusion matrix)
6. Export the model for the app

**Before you start:** upload the whole `carescan` project folder to
`MyDrive/carescan` in Google Drive.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/carescan'   # change if you put it somewhere else
%cd {PROJECT}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics roboflow easyocr
import ultralytics
ultralytics.checks()

## 2. Download datasets from Roboflow

1. Get your API key: Roboflow → Settings → API Keys. **Don't share the notebook with the key in it.**
2. For each public laundry-symbol dataset on [Roboflow Universe](https://universe.roboflow.com)
   (search "laundry symbols" / "care label"), open *Dataset → Download → YOLOv8 → show download code*
   and copy the workspace, project and version into `PUBLIC` below.
3. Put our own labelled project in `OWN`.

In [ ]:
# ONE cell: download all datasets, then merge them. Re-run this whole cell after any change.
import os, sys, subprocess, yaml
from getpass import getpass
from roboflow import Roboflow

os.chdir(PROJECT)   # in case the runtime restarted

# (workspace, project, version) — fill these in, WITHOUT a # at the start of the line
PUBLIC = [
    ('wasabis-workspace', 'care-label-x5z3u-v4l3g', 1),
    ('alexander-pitkin-19mq3', 'laundry-ymnou', 10),     # v10 = bigger, augmented version
]
OWN = [
    ('wasabis-workspace', 'carescan-own-photos', 1),     # raise the version after generating a new one
]
print(f'Datasets listed: {len(PUBLIC)} public, {len(OWN)} own')
assert PUBLIC or OWN, 'PUBLIC and OWN are both empty: fill them in above'

rf = Roboflow(api_key=getpass('Roboflow API key: '))

def download(entries, folder):
    paths = []
    for ws, proj, ver in entries:
        print(f'Downloading {ws}/{proj} v{ver} ...')
        ds = rf.workspace(ws).project(proj).version(ver).download('yolov8', location=f'{folder}/{proj}_v{ver}')
        paths.append(ds.location)
    return paths

public_dirs = download(PUBLIC, '/content/downloads/public')
own_dirs = download(OWN, '/content/downloads/own')
print('\npublic:', public_dirs)
print('own:   ', own_dirs)

for d in public_dirs + own_dirs:
    names = yaml.safe_load(open(f'{d}/data.yaml'))['names']
    print(f'{d}: {len(names)} classes')

cmd = [sys.executable, 'scripts/merge_datasets.py', '--out', '/content/data/merged']
if public_dirs: cmd += ['--public', *public_dirs]
if own_dirs:    cmd += ['--own', *own_dirs]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout, r.stderr)

## 3. Split and check balance

In [ ]:
!python scripts/split_dataset.py --src /content/data/merged --out /content/data/carescan
!python scripts/check_balance.py --data /content/data/carescan --min 50

If many classes are **LOW** or **MISSING**: photograph more labels with those symbols,
or merge rare classes in `configs/class_aliases.yaml` (e.g. `wash_30_gentle: wash_30`).
A first training run with an incomplete dataset is still useful, because it shows which classes struggle.

## 4. Train

Transfer learning: start from `yolov8n.pt` (pretrained on COCO) and fine-tune on our symbols.

Augmentation notes:
- `fliplr=0, flipud=0`: **never flip**, because mirrored symbols are wrong.
- `degrees=180`: random rotation in any direction (labels are photographed at any angle).
  If mAP is poor, try `degrees=30`; big rotations make the boxes looser.
- HSV colour, scale, translate and mosaic augmentations are on by default. Blur and CLAHE come from
  Albumentations automatically if it is installed.
- Care symbols are small: if small symbols are missed, train with `imgsz=960`.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'yolov8n_v2_noleak'   # change for every experiment, keep old results for the report

model = YOLO('yolov8n.pt')       # try 'yolov8s.pt' later for better accuracy
results = model.train(
    data='/content/data/carescan/data.yaml',
    epochs=100,
    patience=20,                 # stop early if val mAP stops improving
    imgsz=640,
    batch=16,
    fliplr=0.0, flipud=0.0,
    degrees=180,
    project=f'{PROJECT}/runs',   # saved on Drive, so it survives Colab disconnects
    name=RUN_NAME,
    seed=42,
)

Training curves (loss should go down, mAP should go up and level off):

In [ ]:
import glob, os
from IPython.display import Image, display
# Ultralytics adds a number to the folder name if it already exists (yolov8n_v12, ...),
# so take the folder the trainer actually used, or else the newest matching folder.
try:
    RUN = str(model.trainer.save_dir)
except (NameError, AttributeError):
    RUN = sorted(glob.glob(f'{PROJECT}/runs/{RUN_NAME}*'), key=os.path.getmtime)[-1]
print('Run folder:', RUN, '->', sorted(os.listdir(RUN)))
display(Image(f'{RUN}/results.png', width=900))

import pandas as pd
hist = pd.read_csv(f'{RUN}/results.csv')
hist.columns = hist.columns.str.strip()
print('Epochs trained:', len(hist))
print(hist[['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)',
            'metrics/mAP50-95(B)']].tail(5).to_string(index=False))

If Colab disconnects, resume with:
```python
YOLO(f'{PROJECT}/runs/{RUN_NAME}/weights/last.pt').train(resume=True)
```

## 5. Evaluate on the test set

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data='/content/data/carescan/data.yaml', split='test', imgsz=640,
             project=f'{PROJECT}/runs', name=f'{RUN_NAME}_test')

print(f'mAP@0.5      : {m.box.map50:.3f}   (target >= 0.85)')
print(f'mAP@0.5:0.95 : {m.box.map:.3f}')
print(f'Precision    : {m.box.mp:.3f}')
print(f'Recall       : {m.box.mr:.3f}')

In [ ]:
# Per-class AP@0.5: find the weakest symbols
import pandas as pd
names = best.names
rows = [(names[c], m.box.ap50[i]) for i, c in enumerate(m.box.ap_class_index)]
df = pd.DataFrame(rows, columns=['class', 'AP@0.5']).sort_values('AP@0.5')
df.to_csv(f'{m.save_dir}/per_class_ap.csv', index=False)
df

In [ ]:
# Confusion matrix: which symbols get mixed up?
display(Image(f'{m.save_dir}/confusion_matrix_normalized.png', width=900))

In [ ]:
# Look at predictions on our own test photos
import glob, random, cv2
own_test = sorted(glob.glob('/content/data/carescan/test/images/own__*'))
for p in random.sample(own_test, min(6, len(own_test))):
    r = best.predict(p, conf=0.25, verbose=False)[0]
    display(Image(data=cv2.imencode('.jpg', r.plot())[1].tobytes(), width=500))
    print(p, '->', [names[int(c)] for c in r.boxes.cls])

## 6. Export for the app

Copies `best.pt` into `models/` so the Streamlit app and evaluation script can use it.

In [ ]:
import shutil, os
os.makedirs('models', exist_ok=True)
# Keep the first model under its own name before overwriting models/best.pt
if os.path.exists('models/best.pt') and not os.path.exists('models/best_yolov8n_v1.pt'):
    shutil.copy('models/best.pt', 'models/best_yolov8n_v1.pt')
    print('Backed up the previous model as models/best_yolov8n_v1.pt')

shutil.copy(f'{RUN}/weights/best.pt', f'models/best_{RUN_NAME}.pt')   # kept for comparison
shutil.copy(f'{RUN}/weights/best.pt', 'models/best.pt')               # the app uses this one
# Optional faster CPU inference:
# best.export(format='onnx', imgsz=640)
print(f'Saved models/best_{RUN_NAME}.pt and models/best.pt')

## 7. (Optional) End-to-end evaluation

Runs the full pipeline (preprocess → YOLO → OCR → rules) on our own photos and compares with
`data/ground_truth.csv`. This measures what the user actually gets, including speed.

In [ ]:
# Needs data/ground_truth.csv and a folder data/own_photos/ — skip until those exist.
if os.path.exists('data/ground_truth.csv') and os.path.isdir('data/own_photos'):
    !python scripts/evaluate_pipeline.py --model models/best.pt --images data/own_photos --truth data/ground_truth.csv
else:
    print('Skipped: create data/ground_truth.csv and data/own_photos/ first (see README).')

## Experiment log (fill in for the report)

| Run | Model | imgsz | degrees | Epochs | Train imgs | mAP@0.5 (test) | Notes |
|---|---|---|---|---|---|---|---|
| yolov8n_v1 | v8n | 640 | 180 | 100 | 3,910 | 0.904 | baseline; augmented copies leaked into test; 4 own photos in test |
| yolov8n_v2_noleak | v8n | 640 | 180 | | | | split keeps augmented copies together |
| yolov8n_v3_960 | v8n | 960 | 180 | | | | bigger images for washtub numbers |
| | | | | | | | |